# Lab 7: Report

## Cold Working & Annealing



The main objective of this lab is to study the plastic deformation of ductile metals and the effect of this deformation on their mechanical properties. In this notebook you will:

- Measure percent cold work (%CW) and determine how the degree of cold working affects hardness.
- Observe the change in hardness that results from annealing at different temperatures.
- Relate the changes in mechanical properties to changes in the microstructure.


Recommended reading: Callister, *Materials Science and Engineering: An Introduction*, Sections 7.10–7.13.

## 1. Load the data into the notebook

The data are supplied as a `.csv` file. If you were given an `.xlsx` workbook, open it in Excel and use **File → Save As → CSV (Comma delimited) (\*.csv)** before continuing.

1. Open the `.csv` file (in Excel or a text editor) and inspect it.
    - Each specimen occupies a block of eight rows (thickness, width, area, %CW, three HRB readings, average HRB)
    - Each column is one **step** of the experiment (pass 0 through 8, plus the annealing steps).
1. Place the `.csv` file in the same folder as this notebook.
1. Update `DATA_FILE` below to match the file name.

Run the cell below to import necessary libraries and define the name of `DATA_FILE`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = "Lab7_Data.csv"    # STUDENT: update the text string to match your .csv file


### 1.1 Find the data in the raw table

Run the code cell below to import and inspect the raw data.
- The numbers along the left edge (**row index**) and across the top (**column index**) are the indices of each value.
- Python starts counting at **0**, so row 0 is the first line of the file.

In the raw table:

- **Row 1** holds the step labels, located in **columns 3 to 13**.
- **Column 0** holds the material name. It is written only on the first row of each material.
- **Column 1** holds the anneal temperature in °C.
- **Column 2** says what each row contains. Every specimen block starts on a row labeled `thickness (in)`.

Within each specimen block, the rows are always in the same order:

To pick values out of a DataFrame by position, use `.iloc[row, column]`. A colon selects a range: `raw.iloc[3, 3:14]` means row 3, columns 3 up to **but not including** 14 (i.e. columns 3 to 13).

Run the cell below to see the thickness of the first specimen at every step. Compare it with the `.csv` file.

In [ ]:
# Read the file exactly as it is laid out
raw = pd.read_csv(DATA_FILE, header=None, dtype=str, encoding="latin-1")

# Remove columns that are completely empty
raw = raw.dropna(axis="columns", how="all")

display(raw)

**NOTE**: Clear the output above before submitting your final report as a .PDF

### 1.2 Reorganize the data into a single table

The `.csv` file comes from an Excel file for easy data entry. For analysis it is easier to have a long table. The cell below builds that table from the `raw` dataframe above. The new long table will be called `data_table`.

| columns made for `data_table` | value |
|:---|:---|
| `step` | order of the measurement (0, 1, 2, …) |
| `step_label` | the step label from row 1 of the file (e.g. `4 (post-an)`) |
| `stage` | `rolled` (measured after a rolling pass) or `annealed` (measured right after a heat treatment) |
| `thickness_in`, `width_in` | caliper measurements in inches |
| `HRB1`, `HRB2`, `HRB3` | the three Rockwell B readings (blank if not measurable) |
| `too_soft` | `True` if the hardness value includes a '0' preceeded by white space |

In [ ]:
# The material name is only written on the first row of each material: copy it down the column.
raw[0] = raw[0].ffill()

# columns 3 to 13 hold the step information (same as 3:14 in .iloc)
STEPS = slice(3, 14)

# The step labels are located in row 1 of the spreadsheet, in columns 3 to 13.  These are the same for all specimens.
step_labels = raw.iloc[1, STEPS]

# Each specimen block starts on the row labeled "thickness (in)" in column 2.
block_starts = raw.index[raw[2] == "thickness (in)"]

# Empty list to hold the six specimen tables.
specimen_tables = []

# iterate through each block_start value, which is the row number of the start of each specimen block
for start in block_starts:
    material = raw.iloc[start, 0]  # extract the material name from column 0 of the start row
    temperature = int(float(raw.iloc[start, 1]))  # extract the temperature from column 1 of the start row, convert to float, then to int

    # pd.to_numeric(..., errors="coerce") converts text to numbers.
    # Anything that is not a number (blank cells, "too soft / graph 0") becomes NaN.
    table = pd.DataFrame({
        "specimen": f"{material}_{temperature} C",
        "step_label": step_labels.to_numpy(),
        "stage": np.where(step_labels.str.contains("post", na=False), "annealed", "rolled"),
        "thickness_in": pd.to_numeric(raw.iloc[start,     STEPS], errors="coerce").to_numpy(),
        "width_in":     pd.to_numeric(raw.iloc[start + 1, STEPS], errors="coerce").to_numpy(),
        "HRB1":         pd.to_numeric(raw.iloc[start + 4, STEPS], errors="coerce").to_numpy(),
        "HRB2":         pd.to_numeric(raw.iloc[start + 5, STEPS], errors="coerce").to_numpy(),
        "HRB3":         pd.to_numeric(raw.iloc[start + 6, STEPS], errors="coerce").to_numpy(),
        "too_soft":     raw.iloc[start + 4, STEPS].str.contains(r"\s+0", na=False).to_numpy(),
    })

    # add the data to the list of specimen tables
    specimen_tables.append(table)


# Stack the six specimen tables on top of each other to make one long table.
data_table = pd.concat(specimen_tables, ignore_index=True)

# Keep only the steps that were actually measured for each specimen.
measured = data_table["thickness_in"].notna() | data_table["HRB1"].notna() | data_table["too_soft"]
data_table = data_table[measured].reset_index(drop=True)

SPECIMENS = list(data_table["specimen"].unique())


print(f"Loaded {len(data_table)} measurements for {len(SPECIMENS)} specimens:")
for s in SPECIMENS:
    print("  ", s)

In [ ]:
# Run this cell to display the data for one specimen.
# SPECIMENS[0] is the first specimen in the list (Python counts from 0); change the index to view the others.

display(data_table[data_table["specimen"] == SPECIMENS[1]])

**`ENGINEERING CHECK`**

Compare the table above with the same specimen in the `.csv` file before continuing. Some values are listed as `NaN`, which stands for "Not a Number".
- `thickness_in` and `width_in`: replace `NaN` with the value just above it
- `HRB` when `too_soft` is `true`: replace `NaN` with `0`.

`YOUR RESPONSE`:

Complete and run the code below to fill in these missing values.

`.ffill()`: replaces any missing values in a column with the value just above it

In [ ]:
# Use the ffill() method to use the pre-anneal dimensions as the post-anneal dimensions
data_table["thickness_in"] = data_table.groupby("specimen")["thickness_in"]  # STUDENT: complete this line to fill in the missing values
data_table["width_in"] = data_table.groupby("specimen")["width_in"]          # STUDENT: complete this line to fill in the missing values

# if too_soft is True, set the HRB values to 0
data_table.loc[data_table["too_soft"], ["HRB1", "HRB2", "HRB3"]] = 0      # STUDENT: complete this line to set NaN HRB values to 0

# print out the data for one specimen to check that the missing values have been filled in correctly.
display(data_table[data_table["specimen"] == SPECIMENS[2]])

## 2. Perform calculations

**`YOUR RESPONSE`**: Write the equation you used to calculate %CW in LaTeX, define every symbol, and state where you found it.

$$
\%\text{CW} = \ldots
$$

In the code below perform all the calculations needed for the rest of the report. Each step adds new columns to `data_table`.

**Functions for your code**:

- `.mean(axis=1)`: average across columns
- `.std(axis=1)`: sample standard deviation across columns
- `.count(axis=1)`: number of non-missing values across columns
- `.round(n)`: round to n decimal places

**Variables for your code**:

- `data_table["thickness_in"]`: specimen thickness
- `data_table["width_in"]`: specimen width
- `data_table["area_in2"]`: cross-sectional area at each step (you calculate this in Step 2)
- `data_table[HRB_COLUMNS]`: the three hardness readings
- `A0`: initial cross-sectional area of each specimen


**Step 1. Calulate Percent Cold Work**



In [ ]:
# calculate cross-sectional area. Round to correct significant figures
data_table["area_in2"] = YOUR RESPONSE

# This line stores the initial cross-sectional area for each specimen
A0 = data_table.groupby("specimen")["area_in2"].transform("first")

# calculate percent cold work. Round to correct significant figures
data_table["%CW"] = STUDENT RESPONSE

# print out the data for one specimen to check that the missing values have been filled in correctly.
display(data_table.head(5))


**Step 2. Calculate the hardness statistics**

The three readings are in three **columns** of the same row, so tell pandas to work across the columns with `axis=1`. Daisy chain the rounding action on the end.

For example: `imported_data["column 1", "column 2", ... etc].mean(axis=1).round(4)`.

In [ ]:
# calculate the hardness statistics. Round to correct significant figures
data_table["mean HRB"] = STUDENT RESPONSE
data_table["std HRB"] = STUDENT RESPONSE

display(data_table)

**`ENGINEERING CHECK`**

Inspect the table above before continuing. Compare your values with the `.csv` file.

## 3. Plot Rockwell B hardness versus %CW for all six specimens


Plot the **average HRB** against **%CW**, with vertical error bars equal to the standard deviation of the three readings. Because an annealed step has the same %CW as the rolled step just before it, an anneal appears on the plot as a **vertical drop** in hardness at constant %CW.

First, set your plotting choices. Give each of the six specimens a unique color, a unique marker, or both. The names in `COLOR` and `MARKER` must match the specimen names printed in Section 1.2.

In [ ]:
# STUDENT: define a color and a linestyle for each of the six specimens. They are all the same by default
COLOR = {
    "copper_23 C": "k",
    "copper_400 C": "k",
    "copper_500 C": "k",
    "brass_23 C": "k",
    "brass_400 C": "k",
    "brass_500 C": "k",
}
LINESTYLE = {
    "copper_23 C": "-",
    "copper_400 C": "-",
    "copper_500 C": "-",
    "brass_23 C": "-",
    "brass_400 C": "-",
    "brass_500 C": "-",
}

# STUDENT: define font sizes for the plot labels and tick labels.
AXIS_LABEL_FONT_SIZE = YOUR RESPONSE
TITLE_LABEL_FONT_SIZE = YOUR RESPONSE

# this line updates the default font sizes for all plots in this notebook with your values
plt.rcParams.update({"axes.titlesize": TITLE_LABEL_FONT_SIZE,
                     "axes.labelsize": AXIS_LABEL_FONT_SIZE,
                    })


### Rockwell B hardness vs. %CW (20 points)

Complete the plotting code below.

**Variables available inside the loop** (for one specimen at a time):

- `data["%CW"]`: x values
- `data["HRB_mean"]`: y values
- `data["HRB_std"]`: vertical error bars

**Functions**:

- `ax.errorbar(x, y, yerr=..., ...)`: plot data with error bars (the formatting arguments are already given)
- `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.set_title()`: axis labels and title

Refer to the plotting tips in the Lab 2 notebook. Finish by writing a caption that tells the reader what is plotted, what the vertical drops are, what the error bars represent, and what the hollow markers mean.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

for specimen in SPECIMENS:
    data = data_table[data_table["specimen"] == specimen].sort_values("step_label")

    # STUDENT: plot the data with errorbars. Use the color and linestyle defined previously
    ax.errorbar(
        YOUR RESPONSE
    )

    # mark readings that were too soft to measure with a hollow marker
    soft = data[data["too_soft"]]
    if not soft.empty:
        ax.scatter(soft["%CW"],
                   soft["mean HRB"],
                   s=110,
                   linestyle=LINESTYLE[specimen],
                   facecolors="white",
                   edgecolors=COLOR[specimen],
                   zorder=5,
                   label="_",
                   )

# one legend entry explaining the hollow markers
ax.scatter([], [],
           s=110,
           marker="o",
           facecolors="white",
           edgecolors="gray",
           label="too soft for HRB (graphed as 0)",
           )

YOUR RESPONSE  # STUDENT: descriptive title
YOUR RESPONSE  # STUDENT: x-axis label with units
YOUR RESPONSE  # STUDENT: y-axis label with units
YOUR RESPONSE  # STUDENT: set x-axis limits
YOUR RESPONSE  # STUDENT: set y-axis limits
YOUR RESPONSE  # STUDENT: set the opacity of the grid lines
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption. Explain the trends in the caption.
print("Figure 1. DESCRIPTIVE CAPTION HERE")

**`ENGINEERING CHECK`**

Inspect your figure before continuing. Can every curve and every point be distinguished? Try different line-styles and colors.

## 4. Microstructure

Micrographs of copper and brass before cold rolling, at ≈25 %CW, and at ≈50 %CW (all without annealing) are provided on D2L.
- Download the six micrographs and place them in the same folder as this notebook.
- Note: the rolling direction is horizontal in all the images.

### Importing and displaying images

Matplotlib can read and display an image file in your Jupyter notebook. Inspect the code below to see how this works.


In [ ]:
# Read one image file and display it
image = plt.imread("Cu_0CW.jpg")      # STUDENT: update the file name to match one of your micrographs


print("Image size (rows, columns, color channels):", image.shape)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(image)
ax.set_title("Copper, 0 %CW")
ax.axis("off")

# add an arrow showing the rolling direction
ax.annotate(
    "Rolling direction",
    fontsize = 12,
    fontstyle = "oblique",
    xy=(.7, .05),
    xytext=(0.25, .03),
    xycoords="axes fraction",
    ha="center",
    va="bottom",
    arrowprops=dict(arrowstyle="->", color="white", linewidth=2),
    color="white",
)

# add text to label the image as "(a)""
ax.annotate(
    "(a)",
    xycoords="axes fraction",
    xy=(.03, .93),
    color="white",
    fontsize=18,
    )
plt.show()

### 4.1 List the micrograph files

Update the file names in `MICROGRAPHS` to match your files and run the cell. The titles on the left are used to label each image.

The entries are listed in order: the three copper micrographs, then the three brass micrographs.

In [ ]:
# STUDENT: update the file names to match the micrographs you downloaded
MICROGRAPHS = {
    "Copper, 0 %CW": "Cu_0CW.jpg",
    "Copper, 25 %CW": "Cu_25CW.jpg",
    "Copper, 50 %CW": "Cu_50CW.jpg",
    "Brass, 0 %CW": "Brass_0CW.jpg",
    "Brass, 25 %CW": "Brass_25CW.jpg",
    "Brass, 50 %CW": "Brass_50CW.jpg",
}

# labels for calling out specific micrographs
fig_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]


### 4.2 Two ways to display the micrographs

There is more than one way to show the same six images. Run both options below and compare them.

**Option A** loops over the files and makes a separate figure for each micrograph. 

It uses `plt.figure()`, `plt.imshow()`, and `plt.title()`, which draw on the current figure without creating a named set of axes first.

In [ ]:
# Option A: a separate figure for each micrograph

for (title, filename), label in zip(MICROGRAPHS.items(), fig_labels):
    image = plt.imread(filename)
    plt.figure()
    plt.imshow(image)
    plt.title(title)
    plt.annotate(
        "Rolling direction",
        fontsize = 12,
        fontstyle = "oblique",
        xy=(.7, .05),
        xytext=(0.25, .03),
        xycoords="axes fraction",
        ha="center",
        va="bottom",
        arrowprops=dict(arrowstyle="->", color="white", linewidth=2),
        color="white",
    )
    plt.annotate(
        label,
        xycoords="axes fraction",
        xy=(.03, .93),
        color="white",
        fontsize=18,
    )
    plt.axis("off")

    plt.show()


# STUDENT: provide a descriptive caption for the series of 6 images
print("Figure 2. PROVIDE A CAPTION.")

**Option B** creates one figure containing a grid of 2 rows × 3 columns of axes, and draws one micrograph on each set of axes.

To show several images side by side in one figure, create a grid of axes with `plt.subplots(rows, columns)` and draw one image on each set of axes. `axes.ravel()` flattens the grid into a simple list so a `for` loop can step through the panels in order: across the first row, then across the second row.


In [ ]:
# Option B: all six micrographs in one figure
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))     # a grid of 2 rows x 3 columns of panels

for ax, (title, filename), label in zip(axes.ravel(), MICROGRAPHS.items(), fig_labels):
    ax.imshow(plt.imread(filename))
    ax.set_title(title)
    ax.axis("off")
    ax.annotate(
        "Rolling direction",
        fontsize = 12,
        fontstyle = "oblique",
        xy=(.7, .05),
        xytext=(0.25, .03),
        xycoords="axes fraction",
        ha="center",
        va="bottom",
        arrowprops=dict(arrowstyle="->", color="white", linewidth=2),
        color="white",
    )
    ax.annotate(
        label,
        xycoords="axes fraction",
        xy=(.03, .93),
        color="white",
        fontsize=18,
    )

plt.tight_layout()
plt.show()

# STUDENT: provide a descriptive caption for the composite image
print("Figure 2. PROVIDE A CAPTION")   # STUDENT: provide a caption


Decide which option presents the micrographs better in a report.

Keep only the option you chose in your submitted report: delete the other code cell, and make sure the figure you keep has a complete caption

`YOUR RESPONSE`:

Explain why you chose either the series of individual images or the composite image. What benefits and sacrifices does you choice result in? How else might you present the images?

*Replace this with your response*

### Microstructural change with increasing %CW (10 points)

**`YOUR RESPONSE`**: Describe the microstructural change with increasing %CW by comparing the micrographs taken in the course of the experiment.

*Replace this text with your response in paragraph form.*

## 5. Effect of composition, extent of cold work, and annealing treatment (30 points)

**`YOUR RESPONSE`**: Using the previous analyses, explain the effect of composition, extent of cold work, and annealing treatment (temperature and time) on the measured hardness.

Address all three factors, and support your explanation with your data (Figure 1 and Figure 2). Call out specific micrographs and relate the changes you see in the micrograph to the processing it has undergone and the hardness values observed.

*Replace this text with your response in paragraph form.*

## CONTRIBUTION STATEMENT

*List each contributor and what they did.*

**`YOUR RESPONSE`**

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. If the PDF compile fails check your $\LaTeX$ code for errors.